# Đồ án cuối kỳ — RNN (LSTM/GRU) và CNN cho phát hiện ransomware trên BitcoinHeist

**Bài toán:** từ các đặc trưng giao dịch của một địa chỉ Bitcoin trong một ngày, dự đoán địa chỉ là `white` hay thuộc họ
ransomware nào → **phân loại 7 lớp**, dữ liệu bảng, mất cân bằng nặng (white 98,6%).

**Ý tưởng chính:** dữ liệu là bảng nên thử **hai cách biểu diễn chuỗi**:
- **A — feature-as-sequence:** 11 đặc trưng của một dòng = chuỗi 11 bước (chia ngẫu nhiên, giống Bài 1).
- **B — address history:** lịch sử ≤ 16 lần xuất hiện gần nhất của cùng địa chỉ = chuỗi thời gian thật (chia theo địa chỉ).

**Mã nguồn** nằm trong gói `final/`: `data.py` (đọc, tiền xử lý, chia, dựng chuỗi), `models.py` (LSTM/GRU, CNN1D, MLP),
`configs.py` (các cấu hình thực nghiệm), `train.py` (huấn luyện + đánh giá), `baselines.py` (XGBoost), `benchmark.py`
(đo tốc độ), `report.py` (bảng + hình). Huấn luyện thật chạy trên GPU (máy Trainbox) bằng
`python -m final.train --rep A` / `--rep B`; notebook này mặc định **nạp kết quả đã lưu** trong `outputs/final/`
(đặt `RETRAIN = True` để huấn luyện lại từ đầu một vài cấu hình ngay trong notebook).

**Nội dung**
1. Đọc & mô tả dữ liệu
2. Tiền xử lý & chia dữ liệu
3. Biểu diễn chuỗi A và B
4. Kiến trúc mô hình
5. Huấn luyện
6. Kết quả biểu diễn A (RNN, CNN, so với baseline)
7. Kết quả biểu diễn B và các thí nghiệm đối chứng
8. Đường cong huấn luyện — overfitting / underfitting
9. Ma trận nhầm lẫn, F1 từng lớp
10. Phân tích lỗi & minh hoạ dự đoán
11. Kết luận

In [ ]:
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Image

from final import data, models, configs, train, report

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_colwidth", 60)
RETRAIN = False                  # True: huấn luyện lại RETRAIN_CONFIGS ngay trong notebook (ghi đè outputs/final)
RETRAIN_CONFIGS = ["A_gru", "A_cnn_bn_do"]
OUT, FIG = train.OUT, report.FIG
print("torch", torch.__version__, "| device", train.DEV)

## 1. Đọc & mô tả dữ liệu

In [ ]:
df, CLASSES = data.load_frame()
print(f"{len(df):,} dòng, {df['address'].nunique():,} địa chỉ, năm {df['year'].min()}–{df['year'].max()}")
print("Giá trị thiếu:", int(df.isna().sum().sum()), "| dòng trùng (address, year, day):",
      int(df.duplicated(['address', 'year', 'day']).sum()))
df.head()

In [ ]:
vc = df["target"].value_counts()
display(pd.DataFrame({"số dòng": vc, "tỉ lệ %": 100 * vc / len(df)}))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
vc.plot.bar(ax=ax[0], logy=True, title="Phân bố 7 lớp (thang log)", color="#4C72B0")
ct = pd.crosstab(df["year"], df["target"] != "white").rename(columns={False: "white", True: "ransomware"})
(100 * ct["ransomware"] / ct.sum(1)).plot.bar(ax=ax[1], title="% dòng ransomware theo năm", color="#DD8452")
plt.tight_layout(); plt.show()

**Nhận xét:** white chiếm 98,6% → accuracy vô nghĩa (đoán toàn white = 98,6%), dùng **macro F1** làm độ đo chính.
Tỉ lệ ransomware thay đổi mạnh theo năm → giữ year/month/weekday làm đầu vào (one-hot).

In [ ]:
# Địa chỉ có xuất hiện lại ở các ngày khác không? → cơ sở của biểu diễn B
hist16, gap, n_prev = data.address_history(df, 16)
df["n_prev"] = n_prev
rep = df.groupby("target").agg(so_dong=("y", "size"), so_dia_chi=("address", "nunique"),
                               lan_dau_pct=("n_prev", lambda s: 100 * (s == 0).mean()),
                               trung_vi_so_lan_truoc=("n_prev", "median"), tb_so_lan_truoc=("n_prev", "mean"))
display(rep.sort_values("so_dong", ascending=False))

**Nhận xét quan trọng:** `CryptoLocker`, `CryptoWall`, `otherRansom` **dùng lại địa chỉ** rất nhiều (chỉ 15–23% dòng là lần
đầu xuất hiện), còn `Cerber`, `Locky` gần như luôn dùng địa chỉ mới (99,5%), giống `white` (90,8%).
Đây đúng là 2 lớp khó nhất ở Bài 1 (F1 ≈ 0,28 / 0,41) → **lịch sử của địa chỉ** có thể giúp phân biệt chúng.

## 2. Tiền xử lý & chia dữ liệu

- Nhãn: 5 họ phổ biến nhất + `otherRansom` + `white`. Đặc trưng: 6 gốc + 5 mới (`income_tz`, các tỉ lệ) → 11 số;
  `year/month/weekday` → one-hot 27 chiều.
- Số: `log1p` → chuẩn hoá (μ, σ **chỉ tính trên train**).
- Chia 64/16/20: **ngẫu nhiên phân tầng** (biểu diễn A, như Bài 1) hoặc **theo địa chỉ** (biểu diễn B).
- Train: undersample còn 100k white + toàn bộ ransomware; sau huấn luyện tìm trọng số lớp `w` trên validation.
- `val_us`: validation undersample cùng tỉ lệ white với train → đường cong train/val so sánh được, dùng cho early stopping.

In [ ]:
prepA = data.prepare("random")
prepB = data.prepare("group", history_len=32)
for name, p in [("A — chia ngẫu nhiên", prepA), ("B — chia theo địa chỉ", prepB)]:
    sizes = {k: len(v) for k, v in p.idx.items()}
    ransom = {k: int((p.y[v] != p.white).sum()) for k, v in p.idx.items()}
    display(pd.DataFrame({"số dòng": sizes, "ransomware": ransom}).T.rename_axis(name))
addr = df["address"].to_numpy()
print("Số địa chỉ chung giữa train và test (chia theo địa chỉ):",
      len(set(addr[np.r_[prepB.idx['fit']]]) & set(addr[prepB.idx['test']])))
print("Kiểm tra chuẩn hoá trên tập fit: mean ≈ 0, std ≈ 1 →",
      prepA.num[prepA.idx['fit']].mean(0).round(3).max(), prepA.num[prepA.idx['fit']].std(0).round(3).min())

## 3. Biểu diễn chuỗi

**A — feature-as-sequence:** một dòng → chuỗi 11 bước; bước *i* là giá trị chuẩn hoá của đặc trưng *i*, được nhúng thành
vector 16 chiều bằng `x_i·W_i + b_i` (W_i, b_i riêng cho từng đặc trưng). Độ dài cố định, không cần đệm.
*Hạn chế:* thứ tự đặc trưng do ta đặt, không có ý nghĩa thời gian.

In [ ]:
i = prepA.idx["test"][np.flatnonzero(prepA.y[prepA.idx["test"]] == CLASSES.index("princetonLocky"))[0]]
display(pd.DataFrame({"đặc trưng (bước)": data.NUM, "giá trị gốc": df.loc[i, data.NUM].to_numpy(),
                      "sau log1p + chuẩn hoá": prepA.num[i]}).T)

**B — address history:** với dòng (địa chỉ *a*, ngày *t*) lấy tối đa *L* lần xuất hiện gần nhất của *a* tại các ngày ≤ *t*,
theo thời gian; bước cuối = dòng hiện tại. Mỗi bước 13 chiều (11 đặc trưng + cờ "lần đầu" + log(1+Δngày)).
Chuỗi dài ngắn khác nhau → đệm 0 ở cuối + mask (`pack_padded_sequence` cho RNN, pooling có mask cho CNN).

In [ ]:
cl = CLASSES.index("montrealCryptoLocker")
cand = prepB.idx["test"][(prepB.y[prepB.idx["test"]] == cl)]
r = cand[np.argmax((prepB.hist[cand] >= 0).sum(1) >= 6)]
h = prepB.hist[r][prepB.hist[r] >= 0][-8:]
ex = df.loc[h, ["address", "year", "day", "income", "length", "count", "neighbors", "target"]].copy()
ex["Δngày"] = np.where(prepB.gap[h] < 0, np.nan, prepB.gap[h])
print("Lịch sử (8 bước cuối) của một địa chỉ CryptoLocker trong tập test — dòng cuối là ngày cần dự đoán:")
display(ex)
L = (prepB.hist >= 0).sum(1)
pd.crosstab(df["target"], np.minimum(L, 16), normalize="index").mul(100).round(1).iloc[:, [0, 1, 2, 4, 9, 15]] \
  .rename(columns=lambda c: f"L={c}" if c < 16 else "L≥16")

## 4. Kiến trúc mô hình

In [ ]:
def show_model(name, prep):
    cfg = configs.ALL[name]
    b = train.Batcher(prep, cfg["rep"], cfg["train"].get("history_len"), cfg["train"].get("side_count", False))
    m = models.build(cfg, b.in_dim, b.side.shape[1], len(CLASSES), b.seq_len)
    print(f"===== {name}: {models.n_params(m):,} tham số | train: {cfg['train']}")
    print(m)

for n in ["A_lstm", "A_cnn_bn_do"]:
    show_model(n, prepA)
for n in ["B_gru", "B_cnn_bn_do"]:
    show_model(n, prepB)

In [ ]:
rows = []
for c in configs.A + configs.B:
    rows.append({"cấu hình": c["name"], "biểu diễn": c["rep"], "loại": c["model"],
                 "kiến trúc": report.describe({**c, "config": c["name"]}),
                 "lr": c["train"]["lr"], "batch": c["train"]["batch_size"], "L": c["train"].get("history_len", "-")})
pd.DataFrame(rows)

**Các thành phần chung:** đầu ra Dense 7 + softmax; hàm mất mát cross-entropy; Adam (lr 1e-3), batch 512, tối đa 100 epoch,
early stopping theo macro F1 trên `val_us` (patience 10), cắt gradient ≤ 1. RNN: dropout 0,2 ở lớp Dense cuối và giữa các
tầng khi có 2 tầng (không dùng recurrent dropout — cuDNN không hỗ trợ). CNN: Conv1D + (BN) + ReLU + (Dropout 0,3) + MaxPool 2,
rồi Flatten (A) hoặc Global Average/Max Pooling có mask (B).

## 5. Huấn luyện

In [ ]:
if RETRAIN:
    for n in RETRAIN_CONFIGS:
        train.run(configs.ALL[n], prepA if configs.ALL[n]["rep"] == "A" else prepB,
                  "random" if configs.ALL[n]["rep"] == "A" else "group")
res = report.load_results()
summ = report.summary_table(res)
print(f"Đã nạp {len(res)} lần chạy, {len(summ)} cấu hình")
bench = json.load(open(OUT / "benchmark.json")) if (OUT / "benchmark.json").exists() else {}

## 6. Kết quả biểu diễn A (chia ngẫu nhiên)

In [ ]:
cols = ["config", "arch", "lr", "batch_size", "n_params", "best_epoch", "epochs", "macro_f1", "macro_f1_std", "n_seeds",
        "binary_f1", "accuracy", "weighted_f1"]
A = summ[summ["group"] == "A_random"]
print("RNN"); display(A[A["model"] == "rnn"][cols])
print("CNN"); display(A[A["model"] == "cnn"][cols])
print("MLP"); display(A[A["model"] == "mlp"][cols])

In [ ]:
# So sánh với mô hình nền của Bài 1 (cùng cách chia ngẫu nhiên, cùng tiền xử lý & hiệu chỉnh trọng số lớp)
b1 = report.bai1_baselines().assign(nguồn="Bài 1")
best = lambda m: A[A["model"] == m].iloc[0]
nn_rows = pd.DataFrame([{"model": f"{best(m)['config']} ({m.upper()})", "macro_f1": best(m)["macro_f1"],
                         "binary_f1": best(m)["binary_f1"], "accuracy": best(m)["accuracy"],
                         "weighted_f1": best(m)["weighted_f1"], "train_time_s": best(m)["train_time_s"],
                         "nguồn": "Đồ án"} for m in ["rnn", "cnn", "mlp"]])
pd.concat([b1, nn_rows]).sort_values("macro_f1", ascending=False).reset_index(drop=True)

In [ ]:
display(Image(FIG / 'bars_A.png'))

In [ ]:
# Tốc độ huấn luyện đo riêng (GPU không chia sẻ): giây/epoch trên ~126.500 dòng, batch 512
pd.DataFrame(bench).T.rename(columns={"time_per_epoch_s": "giây/epoch", "n_params": "tham số"}) if bench else "chưa có benchmark.json"

## 7. Kết quả biểu diễn B (chia theo địa chỉ) và các thí nghiệm đối chứng

In [ ]:
B = summ[summ["group"].isin(["B_group", "A_group", "baseline_group"])]
B[["group"] + cols]

In [ ]:
display(Image(FIG / 'bars_B.png'))

In [ ]:
# F1 nhị phân trên test theo số lần địa chỉ đã xuất hiện (tính cả ngày hiện tại): mô hình học được gì từ lịch sử?
report.by_len_table(prepB).round(3)

## 8. Đường cong huấn luyện — overfitting / underfitting

In [ ]:
for f in ["curves_A_rnn.png", "curves_A_cnn.png", "curves_A_reg.png", "curves_A_lr.png", "curves_B.png", "curves_B_len.png"]:
    if (FIG / f).exists():
        display(Image(FIG / f))

In [ ]:
# Khoảng cách train – val (val_us) ở epoch được chọn và ở epoch cuối — dấu hiệu overfitting / underfitting
g = report.gap_table()
g["chênh F1 @best"] = g["train_f1_best"] - g["val_f1_best"]; g["chênh F1 @cuối"] = g["train_f1_last"] - g["val_f1_last"]
g.round(3)

## 9. Ma trận nhầm lẫn & F1 từng lớp

In [ ]:
for f in sorted(FIG.glob("cm_*.png")):
    display(Image(f))

In [ ]:
pc = []
for _, r in res[res.apply(lambda r: (r["group"], r["name"]) in {(g, n) for g, n, _ in report.key_models(summ)}, axis=1)].iterrows():
    pc.append({"mô hình": f"{r['name']} ({r['split']})", **{report.SHORT[c]: v["f1-score"] for c, v in r["per_class"].items()}})
pd.DataFrame(pc).set_index("mô hình")

## 10. Phân tích lỗi & minh hoạ dự đoán

In [ ]:
def load_model(group, name):
    ck = torch.load(OUT / group / name / "model.pt", map_location="cpu", weights_only=False)
    m = models.build(ck["cfg"], ck["in_dim"], 27 + 3 * bool(ck["cfg"]["train"].get("side_count")), len(CLASSES), ck["seq_len"])
    m.load_state_dict(ck["state_dict"]); m.eval()
    return m, ck

def predict_rows(group, name, prep, rows):
    m, ck = load_model(group, name)
    b = train.Batcher(prep, ck["cfg"]["rep"], ck["cfg"]["train"].get("history_len"), ck["cfg"]["train"].get("side_count", False))
    with torch.no_grad():
        P = torch.softmax(m(*b(rows)[:3]), 1).numpy()
    S = P * ck["class_weights"]
    return P, S / S.sum(1, keepdims=True)

dev0 = train.DEV
train.DEV = torch.device("cpu")      # suy luận vài dòng trên CPU
rng = np.random.RandomState(0)
te = prepB.idx["test"]
rows = np.concatenate([rng.choice(te[prepB.y[te] == k], 2, replace=False) for k in range(len(CLASSES))])
P, S = predict_rows("B_group", "B_gru", prepB, rows)
demo = df.loc[rows, ["address", "year", "day", "income", "count", "neighbors", "target"]].copy()
demo["số lần trước"] = prepB.n_prev[rows]
demo["dự đoán"] = [CLASSES[k] for k in S.argmax(1)]
demo["độ tin cậy"] = S.max(1)
demo["đúng?"] = demo["dự đoán"] == demo["target"]
demo

In [ ]:
# Các lỗi phổ biến nhất của mô hình B tốt nhất và của A tốt nhất (cặp thực tế → dự đoán)
def top_errors(group, name, prep, k=8):
    p = np.load(OUT / group / name / "pred_test.npy"); y = prep.y[prep.idx["test"]]
    e = pd.Series([f"{report.SHORT[CLASSES[a]]} → {report.SHORT[CLASSES[b]]}" for a, b in zip(y[p != y], p[p != y])])
    return e.value_counts().head(k).rename(f"{name}: số lỗi")

bestA = A[A["model"] == "rnn"].iloc[0]["config"]
bestB = summ[(summ["group"] == "B_group") & (summ["model"] == "rnn")].iloc[0]["config"]
display(pd.concat([top_errors("A_random", bestA, prepA).reset_index(), top_errors("B_group", bestB, prepB).reset_index()], axis=1))

In [ ]:
# Dòng ransomware bị đoán là white: đặc trưng trông như thế nào so với ransomware được phát hiện đúng?
p = np.load(OUT / "B_group" / bestB / "pred_test.npy"); te = prepB.idx["test"]; y = prepB.y[te]
miss = te[(y != prepB.white) & (p == prepB.white)]; hit = te[(y != prepB.white) & (p == y)]
feat = df[data.NUM[:6]].assign(n_prev=prepB.n_prev)
cmp = pd.DataFrame({"bỏ sót (→ white)": feat.iloc[miss].median(), "phát hiện đúng": feat.iloc[hit].median(),
                    "white (trung vị)": feat.iloc[te[y == prepB.white]].median()})
print(f"Bỏ sót {len(miss):,} / {(y != prepB.white).sum():,} dòng ransomware; theo lớp:")
display(pd.Series([report.SHORT[CLASSES[k]] for k in prepB.y[miss]]).value_counts().to_frame("bỏ sót"))
cmp

In [ ]:
# Recall từng họ ransomware: dòng địa chỉ xuất hiện LẦN ĐẦU vs dòng LẶP LẠI (mô hình B tốt nhất vs XGBoost + lịch sử)
first = prepB.n_prev[te] == 0
rows_ = []
for g_, n_ in [("B_group", "B_cnn_k5"), ("B_group", bestB), ("baseline_group", "XGB_hist")]:
    p_ = np.load(OUT / g_ / n_ / "pred_test.npy")
    for k, c in enumerate(CLASSES):
        if k != prepB.white:
            m = y == k
            rows_.append({"mô hình": n_, "lớp": report.SHORT[c], "% lần đầu": 100 * first[m].mean(),
                          "recall lần đầu": (p_[m & first] == k).mean(), "recall lặp lại": (p_[m & ~first] == k).mean()})
pd.DataFrame(rows_).pivot(index="lớp", columns="mô hình", values=["recall lần đầu", "recall lặp lại"]).round(3)

In [ ]:
train.DEV = dev0

## 11. Kết luận

Chi tiết trong báo cáo (`report/main.pdf`, mục 6–8). Tóm tắt:

- **Biểu diễn A** (11 đặc trưng → chuỗi 11 bước, chia ngẫu nhiên): RNN đạt macro F1 0,51–0,54, CNN 0,50–0,53 — ngang **MLP**
  (0,52) và **thấp hơn XGBoost của Bài 1 (0,575)**. Thứ tự đặc trưng không có ý nghĩa nên cấu trúc chuỗi không giúp; đổi kiến trúc
  (LSTM/GRU, số tầng, hai chiều, attention, BN, kernel…) chỉ thay đổi ≤ 0,03, phần lớn nằm trong dao động seed (≈ 0,005).
- **Biểu diễn B** (lịch sử ≤ 16 lần xuất hiện của địa chỉ, chia theo địa chỉ): macro F1 tăng lên **0,72–0,74**. Mạng nơ-ron tốt nhất là
  **CNN kernel 5 (`B_cnn_k5`, 0,742)**, RNN tốt nhất `B_gru_2l` (0,735 ± 0,004). Lợi ích đến từ các họ dùng lại địa chỉ
  (CryptoLocker 0,26 → 0,70; CryptoWall 0,38 → 0,84; otherRansom 0,08 → 0,66).
- **XGBoost + đặc trưng lịch sử tổng hợp (0,767)** vẫn tốt nhất toàn bộ: RNN/CNN vượt XGBoost trên địa chỉ có lịch sử dài
  (≥ 6 lần xuất hiện) nhưng thua trên địa chỉ mới — nhóm chiếm 90% dòng test và 53% dòng ransomware.
- **Overfitting:** A gần như không overfit (giới hạn do thông tin), điều chuẩn thêm chỉ gây underfit; B overfit rõ hơn
  (val loss tăng từ sớm) và được lợi từ BatchNorm + Dropout (CNN 0,686 → 0,731) và dropout giữa các tầng GRU.
- **Tốc độ** (đo riêng): MLP nhanh nhất; ở A, RNN 1 tầng ≈ CNN (≈ 0,4 s/epoch); ở B, CNN nhanh hơn RNN ≈ 25%,
  Bi-GRU chậm nhất (gấp 2,4 lần GRU).
- Hạn chế: lớp white được lấy mẫu 1.000 dòng/ngày nên tín hiệu "địa chỉ lặp lại" có thể lạc quan hơn thực tế.